This notebook runs a shared-start (canonical) one-step comparison between the MalthusJAX benchmark engine and the Evosax benchmark engine. It helps visualize whether both engines begin from the same initial population and how their first generation behaves.


In [1]:
# Canonical One-Step MalthusJAX vs Evosax Comparison
import sys

import jax
import jax.numpy as jnp
import jax.random as jr

sys.path.append('..')
from malthusjax.engine.schedules import TrackBest
from tests.benchmarks.conftest_benchmarks import (
    _build_evosax_ga,
    _build_malthusjax_engine,
    _canonical_population,
)

seed = 42
pop_size = 50
dims = 10
problem = 'ellipsoidal_rotated'
print('Setup complete')

Setup complete


## Configure Engines
We configure both engines with canonical initialization so they start from the same population. MalthusJAX is also set to use the Evosax operator wrappers for a closer parity comparison.


In [2]:
shared_key = jr.PRNGKey(seed)
pop_key, mjx_key, esx_key = jr.split(shared_key, 3)

shared_genes = _canonical_population(pop_key, pop_size, dims)
print('Shared genes shape:', shared_genes.shape)
print('Shared genes first row:', shared_genes[0])

# Build MalthusJAX engine with Evosax-wrapped operators
mjx_engine = _build_malthusjax_engine(
    pop_size=pop_size,
    dims=dims,
    problem=problem,
    num_generations=1000,
    track_best=TrackBest.NONE,
    use_evosax_ops=True,
)

# Build Evosax engine components from the same canonical seed.
# This captures the strategy, problem, and default parameters used
# for the subsequent Evosax state initialization.
esx_strategy, esx_params, es_problem, es_carry = _build_evosax_ga(
    pop_size,
    dims,
    problem,
    rng=esx_key,
    init_x=shared_genes,
)
es_state, p_state, rng_es = es_carry
print('Built and initialized Evosax engine from canonical population')

# Initialize MalthusJAX state and replace with canonical population
state_mjx = mjx_engine.init_state(mjx_key)
genes_cls = type(state_mjx.population.genes)
population = state_mjx.population.replace(genes=genes_cls.from_tensor(shared_genes))
population = mjx_engine.evaluator.evaluate_population(population)
best_idx = jnp.argmax(population.fitness)
best_genome = jax.tree_util.tree_map(lambda x: x[best_idx], population.genes)
state_mjx = state_mjx.replace(
    population=population,
    best_fitness=population.fitness[best_idx],
    best_genome=best_genome,
)

print('Engines initialized with canonical population')

Shared genes shape: (50, 10)
Shared genes first row: [ 0.302608  -1.8663788  4.0153027  1.983329   1.180656   1.2185204
  2.1774673 -2.902999  -4.76612    2.387607 ]


W0407 13:30:11.790077 1951568 cpp_gen_intrinsics.cc:74] Empty bitcode string provided for eigen. Optimizations relying on this IR will be disabled.


Built and initialized Evosax engine from canonical population


/var/folders/n8/08b2nd114jdfnsydb_4mj4fw0000gn/T/ipykernel_45588/2576453572.py:32: DeprecationWarning: Legacy PRNGKey detected in GeneticEngine.init_state(). For explicit PRNG backend control use malthusjax.core.random.create_key() or jax.random.key()
  state_mjx = mjx_engine.init_state(mjx_key)


Engines initialized with canonical population


## Run Multi-Generation Comparison
Run several generations on both engines from the shared canonical start and compare the reported best/mean fitness per generation.

In [4]:
num_gens = 5
print(f'Running {num_gens} generations for both engines')

mjx_history = [
    {
        'generation': 0,
        'best_fitness': float(state_mjx.best_fitness),
        'mean_fitness': float(jnp.mean(state_mjx.population.fitness)),
    }
]
esx_history = [
    {
        'generation': 0,
        'best_fitness': float(es_state.best_fitness),
        'mean_fitness': float(jnp.mean(es_state.fitness)),
    }
]

state_mjx_curr = state_mjx
es_state_curr = es_state
p_state_curr = p_state
rng_es_curr = rng_es

for generation in range(1, num_gens + 1):
    state_mjx_curr, mjx_metrics = mjx_engine.step(state_mjx_curr)
    mjx_history.append(
        {
            'generation': generation,
            'best_fitness': float(state_mjx_curr.best_fitness),
            'mean_fitness': float(jnp.mean(state_mjx_curr.population.fitness)),
        }
    )

    rng_es_curr, rng_es_step = jr.split(rng_es_curr)
    x, es_state_curr = esx_strategy.ask(rng_es_step, es_state_curr, esx_params)
    fitness, p_state_curr, _ = es_problem.eval(rng_es_step, x, p_state_curr)
    es_state_curr, _ = esx_strategy.tell(rng_es_step, x, fitness, es_state_curr, esx_params)
    esx_history.append(
        {
            'generation': generation,
            'best_fitness': float(es_state_curr.best_fitness),
            'mean_fitness': float(jnp.mean(es_state_curr.fitness)),
        }
    )

print('\nMalthusJAX history:')
for entry in mjx_history:
    print(f"gen={entry['generation']} best={entry['best_fitness']:.6e} mean={entry['mean_fitness']:.6e}")

print('\nEvosax history:')
for entry in esx_history:
    print(f"gen={entry['generation']} best={entry['best_fitness']:.6e} mean={entry['mean_fitness']:.6e}")

print('\nNote: Evosax fitness is lower-is-better, while MalthusJAX reports maximization-style fitness with BBOB minimize flipped internally.')

Running 5 generations for both engines

MalthusJAX history:
gen=0 best=-4.644107e+05 mean=-1.060602e+07
gen=1 best=-4.644107e+05 mean=-5.228908e+06
gen=2 best=-4.644107e+05 mean=-4.564982e+06
gen=3 best=-4.644107e+05 mean=-3.534955e+06
gen=4 best=-4.644107e+05 mean=-2.773437e+06
gen=5 best=-4.644107e+05 mean=-2.911318e+06

Evosax history:
gen=0 best=inf mean=inf
gen=1 best=6.029688e+06 mean=2.522516e+07
gen=2 best=3.296256e+06 mean=1.510655e+07
gen=3 best=1.686248e+06 mean=1.021192e+07
gen=4 best=1.514496e+06 mean=6.776650e+06
gen=5 best=6.328908e+05 mean=5.332054e+06

Note: Evosax fitness is lower-is-better, while MalthusJAX reports maximization-style fitness with BBOB minimize flipped internally.


## Interpretation
If both engines share the same initial population and are configured with matching operator semantics, then any difference in their first-step output indicates a remaining mismatch in engine state handling, update ordering, or evaluation bookkeeping.
